# Lección 4.6 — Validación cruzada: k-fold, estratificada y temporal

> **Módulo 4 · Evaluación: ¿mi modelo es bueno?** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 2.5 (entrenamiento, validación y prueba), Lección 2.6 (fuga de datos), Lecciones 4.1–4.3 (métricas), Lección 4.5 (sesgo-varianza)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué evaluar con **una sola división** de los datos es una lotería.
- Usar la **validación cruzada k-fold** y leer su resultado como "promedio ± variación".
- Elegir la variante correcta: **estratificada** (clases desbalanceadas), **temporal** (datos con fecha) o **por grupos** (varias filas de la misma persona).
- Elegir hiperparámetros con validación cruzada **sin tocar** el conjunto de prueba.
- Reconocer las fugas de datos más comunes al validar.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (train_test_split, cross_val_score, cross_validate,
                                     KFold, StratifiedKFold, TimeSeriesSplit, GroupKFold,
                                     StratifiedGroupKFold, GroupShuffleSplit, GridSearchCV)

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: un examen de 10 preguntas es una lotería

Imagina que un maestro decide la calificación final de todo el semestre con **un solo examen de 10 preguntas** elegidas al azar.

- Si te tocan justo los temas que mejor dominas, sacas 10.
- Si te tocan los que menos estudiaste, sacas 6.

Tu conocimiento es el mismo; **la nota depende de la suerte** de qué preguntas tocaron. Un maestro más justo haría **varios exámenes** sobre temas distintos y **promediaría**. Además, vería cuánto **varían** tus notas: si sacas 9, 9, 8, 9, 9, confía en el promedio; si sacas 10, 5, 9, 6, 10, no tanto.

Evaluar un modelo con **una sola división** entrenamiento/prueba es el examen de 10 preguntas. La **validación cruzada** es la serie de exámenes.

### ✋ Ejercicio 1 — Calentamiento
Dos estudiantes tienen el mismo promedio, 8.5, en 5 exámenes:
- Ana: 8.5, 8.4, 8.6, 8.5, 8.5
- Beto: 10, 6.5, 9.5, 7, 9.5

¿A quién le confiarías un examen final **decisivo**? ¿Qué información te da la variación que el promedio no te da?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

A **Ana**. Su promedio es igual de alto, pero **predecible**: sabes casi seguro qué sacará. Con Beto, el resultado del examen final puede ser cualquier cosa entre 6.5 y 10.

Con los modelos pasa igual: el **promedio** te dice qué tan bueno es y la **variación** (desviación estándar) te dice **cuánto confiar** en ese promedio. Siempre se reportan las dos cosas.
</details>

---
## 2. 📖 El concepto: una división es una lotería

Comprobémoslo con datos reales: diagnóstico de cáncer de mama (569 pacientes). Evaluaremos **el mismo modelo** (un árbol de decisión pequeño; lo verás a fondo en la Lección 5.4) con **20 divisiones distintas** de los datos: 80 % para entrenar, 20 % para evaluar. Lo único que cambia es la semilla (`random_state`) que decide qué pacientes caen en cada lado.

### 🔮 Ejercicio 2 — ¿Cuánto cambia la nota?
Antes de ejecutar: ¿crees que la exactitud cambiará **menos de 1 punto**, **unos 3 puntos** o **más de 10 puntos** entre la mejor y la peor división?

✍️ *Mi apuesta:*

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
modelo = DecisionTreeClassifier(max_depth=3, random_state=0)

notas = []
for semilla in range(20):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=semilla)
    notas.append(modelo.fit(X_tr, y_tr).score(X_te, y_te))

notas = np.array(notas)
print("Exactitud con cada división:", np.round(notas, 3))
print(f"Peor: {notas.min():.1%}   Mejor: {notas.max():.1%}   Diferencia: {(notas.max() - notas.min()) * 100:.1f} puntos")

<details><summary><b>👀 Ver respuesta</b></summary>

¡**Más de 10 puntos**! El mismo modelo saca desde **85.1 %** hasta **96.5 %** según qué 114 pacientes cayeron en la prueba.

Imagina que comparas dos modelos con una sola división: uno saca 93 % y el otro 91 %. ¿Es mejor el primero, o solo tuvo una división con suerte? **Con una sola división no lo puedes saber.**
</details>

### 📖 Validación cruzada k-fold
La solución: **rotar** el papel de examen entre todos los datos.

1. Divide los datos en **k** partes iguales llamadas **pliegues** (*folds*). Lo típico es k = 5.
2. Entrena con k − 1 pliegues y evalúa con el pliegue que quedó fuera.
3. Repite **k veces**, dejando fuera un pliegue distinto cada vez.
4. Reporta el **promedio** de las k notas y su **desviación estándar**.

```
              pliegue 1  pliegue 2  pliegue 3  pliegue 4  pliegue 5
Ronda 1:      [ PRUEBA ] [ entrena ] [ entrena ] [ entrena ] [ entrena ]
Ronda 2:      [ entrena ] [ PRUEBA ] [ entrena ] [ entrena ] [ entrena ]
Ronda 3:      [ entrena ] [ entrena ] [ PRUEBA ] [ entrena ] [ entrena ]
Ronda 4:      [ entrena ] [ entrena ] [ entrena ] [ PRUEBA ] [ entrena ]
Ronda 5:      [ entrena ] [ entrena ] [ entrena ] [ entrena ] [ PRUEBA ]
```

Así **cada dato se usa para evaluar exactamente una vez** y para entrenar k − 1 veces. Ningún dato se queda sin "examinar".

▶️ Mira cómo reparte `KFold` 10 filas en 5 pliegues (con `shuffle=True` las baraja antes de repartir):

In [ ]:
filas = np.arange(10)
kf = KFold(n_splits=5, shuffle=True, random_state=0)     # ✏️ cambia n_splits a 2 o 10
for ronda, (entrena, prueba) in enumerate(kf.split(filas), start=1):
    print(f"Ronda {ronda}:  entrena con {entrena}   evalúa con {prueba}")

### ✋ Ejercicio 3 — Cuentas rápidas
Tienes **1,000** filas y usas validación cruzada con **k = 5**.
1. ¿Cuántos modelos se entrenan?
2. ¿Con cuántas filas se entrena cada modelo y con cuántas se evalúa?
3. ¿Cuántas veces se usa para **evaluar** cada fila?
4. ¿Y si usas k = 10?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **5** modelos (uno por ronda).
2. Cada uno se entrena con **800** filas y se evalúa con **200**.
3. **Una** vez cada fila.
4. Con k = 10: **10** modelos, entrenados con **900** filas y evaluados con **100**. Cada fila sigue evaluándose una sola vez, pero el cálculo cuesta el doble.
</details>

---
## 3. 📖 Cómo se usa: `cross_val_score` y `cross_validate`

En scikit-learn todo el proceso es **una línea**: `cross_val_score` hace las k rondas por ti y devuelve las k notas.

### 🔮 Ejercicio 4
En la Sección 2, una sola división daba entre 85 % y 96.5 %. ¿Cuánto crees que valdrá la **desviación estándar** de las 5 notas de la validación cruzada: alrededor de 0.2 puntos, 2 puntos o 10 puntos?

✍️ *Mi apuesta:*

In [ ]:
cv5 = KFold(n_splits=5, shuffle=True, random_state=0)
notas_cv = cross_val_score(modelo, X, y, cv=cv5, scoring="accuracy")

print("Exactitud en cada pliegue:", np.round(notas_cv, 3))
print(f"Resultado: {notas_cv.mean():.1%} ± {notas_cv.std():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

La desviación estándar es de unos **2 puntos** (≈93.7 % ± 1.7 %). Las notas de cada pliegue siguen variando (de ≈91 % a ≈96 %), ¡es la misma lotería! La diferencia es que ahora **lo ves** y lo resumes con un promedio de 5 exámenes, que es mucho más estable que uno solo.

Cómo reportarlo: *"exactitud de 93.7 % ± 1.7 % en validación cruzada de 5 pliegues"*. Si otro modelo saca 94.0 % ± 1.8 %, la diferencia (0.3 puntos) es mucho menor que la variación: **no puedes decir que uno sea mejor**.
</details>

### 📖 Varias métricas a la vez
En las Lecciones 4.2 y 4.3 viste que la exactitud no lo es todo. `cross_validate` calcula **varias métricas** en las mismas rondas. Aquí la clase positiva (1) es "benigno"; para un médico importaría sobre todo el *recall* de los tumores malignos, pero la idea es la misma.

In [ ]:
resultados = cross_validate(modelo, X, y, cv=cv5,
                            scoring=["accuracy", "precision", "recall", "f1"],
                            return_train_score=True)
tabla = pd.DataFrame(resultados).drop(columns=["fit_time", "score_time"])
print(tabla.round(3))
print()
print(tabla.agg(["mean", "std"]).round(3).T)

### ✋ Ejercicio 5 — Lee la tabla
1. Compara `train_accuracy` con `test_accuracy`. ¿Hay señales de sobreajuste fuerte (Lección 4.4)?
2. ¿Qué métrica es la **más estable** entre pliegues (menor desviación)?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Entrenamiento ≈97.2 % y validación ≈93.7 %: una brecha de unos 3.5 puntos. Es una brecha **moderada**, normal en un árbol de 3 niveles; no es el ~100 % vs. ~75 % del sobreajuste de manual. `return_train_score=True` es la forma rápida de hacer el diagnóstico sesgo/varianza de la Lección 4.5 **con validación cruzada**.
2. La **F1** (≈0.950 ± 0.013), seguida de la exactitud (± 0.019). La **precisión** y el ***recall*** son los que más se mueven entre pliegues (± 0.034 y ± 0.035): el pliegue 5, por ejemplo, tiene precisión perfecta (1.000) pero el *recall* más bajo (0.900). Es el tira y afloja de la Lección 4.3, que aparece incluso entre pliegues.

(Detalle técnico: la tabla de abajo usa la desviación estándar de pandas, que da un número un poco mayor que el `± 1.7 %` de la celda anterior; las dos son válidas.)
</details>

### 📖 ¿Cuántos pliegues? Elegir k
| k | Ventaja | Desventaja |
|---|---|---|
| 2–3 | Muy rápido | Cada modelo entrena con poco (50–67 %): la estimación sale **pesimista** |
| **5 o 10** | El estándar: buen equilibrio | — |
| n (uno por fila, *leave-one-out*) | Entrena casi con todo | Lentísimo con muchos datos y su promedio puede ser muy variable |

### 🔮 Ejercicio 6
Al subir k de 2 a 20, cada pliegue de evaluación es **más pequeño**. ¿Qué crees que pasará con la desviación estándar **entre pliegues**?

✍️ *Mi predicción:*

In [ ]:
for k in [2, 5, 10, 20]:     # ✏️ prueba 3 o 50
    notas_k = cross_val_score(modelo, X, y, cv=KFold(n_splits=k, shuffle=True, random_state=0))
    print(f"k = {k:>2}  →  {notas_k.mean():.1%} ± {notas_k.std():.1%}   (cada pliegue evalúa ~{len(y) // k} pacientes)")

<details><summary><b>👀 Ver respuesta</b></summary>

La desviación **sube** con k: de ≈0.5 puntos (k = 2) a ≈5 puntos (k = 20). Con pliegues pequeños, cada nota individual es un "examen" de pocas preguntas: más lotería. El **promedio**, en cambio, se mueve poco (≈92–94 %).

No te asustes por eso: lo que usas para decidir es el **promedio**, que es estable. k = 5 o 10 es casi siempre la elección correcta.
</details>

---
## 4. 📖 Validación cruzada estratificada: cuando una clase es rara

Si solo el **8 %** de tus datos es de la clase positiva (pacientes con una enfermedad rara, máquinas que fallan…), un pliegue elegido al azar puede quedarse con **muy pocos** positivos, o con **ninguno**. En ese pliegue, el *recall* ni siquiera se puede calcular, y la nota de cada pliegue mide cosas distintas.

La **validación estratificada** reparte los datos de modo que **cada pliegue tenga la misma proporción de cada clase** que el conjunto completo.

### 🔮 Ejercicio 7
Tenemos 100 pacientes, 8 con la enfermedad. Con 5 pliegues, ¿cuántos positivos esperas en cada pliegue con `KFold` (al azar)? ¿Y con `StratifiedKFold`?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(0)
y_rara = np.zeros(100, dtype=int)
y_rara[rng.choice(100, size=8, replace=False)] = 1        # 8 positivos en posiciones al azar
X_rara = rng.normal(size=(100, 3))                        # características de relleno

for nombre, cv in [("KFold          ", KFold(5, shuffle=True, random_state=1)),
                   ("StratifiedKFold", StratifiedKFold(5, shuffle=True, random_state=1))]:
    positivos = [int(y_rara[prueba].sum()) for _, prueba in cv.split(X_rara, y_rara)]
    print(f"{nombre} → positivos en cada pliegue de evaluación: {positivos}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `KFold` al azar: **[0, 2, 1, 1, 4]**. ¡El primer pliegue no tiene **ningún** positivo! En esa ronda el *recall* no se puede calcular, y el último pliegue concentra la mitad de los enfermos.
- `StratifiedKFold`: **[1, 1, 2, 2, 2]**. Lo más parejo posible (8 entre 5 no da exacto): cada pliegue es una "mini versión" del dataset completo.

Con clases desbalanceadas (Lección 2.7), la validación estratificada no es un lujo: es la que hace que las notas de cada pliegue sean comparables.
</details>

> 💡 **Buena noticia:** si le pasas a `cross_val_score` un **clasificador** y un número (`cv=5`), scikit-learn usa `StratifiedKFold` automáticamente. Ojo: en ese caso **no baraja** las filas. Si tus datos vienen ordenados (por ejemplo, todos los positivos al final), pásale un `StratifiedKFold(5, shuffle=True, random_state=0)` explícito.

---
## 5. 📖 Validación temporal: no se vale ver el futuro

Una tienda en línea quiere predecir sus **ventas diarias**. Tiene dos años de historia. El negocio tiene una **tendencia** que sube y baja con el tiempo (temporadas, competencia, campañas), además del efecto del fin de semana y de la temperatura.

En la vida real, el modelo se entrenará con el **pasado** y se usará para predecir el **futuro**. Si al validar barajamos los días, el modelo entrena con el martes y el jueves y lo evaluamos con el miércoles: **le estamos dejando ver el futuro**. En producción nunca tendrá esa ventaja.

▶️ Mira los datos:

In [ ]:
rng = np.random.default_rng(0)
n_dias = 730
dia = np.arange(n_dias)
dia_semana = dia % 7                                           # 5 y 6 = fin de semana
tendencia = 200 + np.cumsum(rng.normal(0, 4, n_dias))         # sube y baja sin patrón fijo
temperatura = 20 + 8 * np.sin(2 * np.pi * dia / 365) + rng.normal(0, 2, n_dias)
ventas = tendencia + 15 * (dia_semana >= 5) + 2 * (temperatura - 20) + rng.normal(0, 8, n_dias)

tienda = pd.DataFrame({"dia": dia, "dia_semana": dia_semana, "temperatura": temperatura.round(1),
                       "ventas": ventas.round(0)})
plt.plot(tienda.dia, tienda.ventas, lw=0.8)
plt.xlabel("día"); plt.ylabel("ventas"); plt.title("Ventas diarias: dos años"); plt.show()
tienda.head()

### 📖 `TimeSeriesSplit`: entrenar con el pasado, evaluar con lo que sigue
En lugar de rotar pliegues al azar, la validación temporal **avanza en el tiempo**: cada ronda entrena con todo lo anterior y evalúa con el bloque siguiente.

```
Ronda 1:  [ entrena ][ PRUEBA ]
Ronda 2:  [ entrena  entrena ][ PRUEBA ]
Ronda 3:  [ entrena  entrena  entrena ][ PRUEBA ]
...                                          tiempo ──►
```
▶️ Así reparte `TimeSeriesSplit` nuestros 730 días:

In [ ]:
ts = TimeSeriesSplit(n_splits=5)          # ✏️ prueba gap=7 (deja una semana de "hueco")
for ronda, (entrena, prueba) in enumerate(ts.split(tienda), start=1):
    print(f"Ronda {ronda}: entrena días {entrena.min():>3}–{entrena.max():>3}   evalúa días {prueba.min():>3}–{prueba.max():>3}")

### 🔮 Ejercicio 8 — ¿Cuál es más optimista?
Vamos a evaluar el **mismo modelo** (un bosque aleatorio, Módulo 6; aquí da igual cuál sea) con el error absoluto medio (**MAE**, Lección 4.1) de dos formas:
- **KFold barajado** (ignora el tiempo)
- **TimeSeriesSplit** (respeta el tiempo)

¿Cuál dará **menor** error? ¿Por mucho o por poco? ¿Cuál se parecerá más a lo que pasará en producción?

✍️ *Mi predicción:*

In [ ]:
X_t = tienda[["dia", "dia_semana", "temperatura"]]
y_t = tienda["ventas"]
bosque = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)

for nombre, cv in [("KFold barajado ", KFold(5, shuffle=True, random_state=0)),
                   ("TimeSeriesSplit", TimeSeriesSplit(5))]:
    mae = -cross_val_score(bosque, X_t, y_t, cv=cv, scoring="neg_mean_absolute_error")
    print(f"{nombre} → MAE por pliegue: {np.round(mae, 1)}   promedio: {mae.mean():.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El KFold barajado dice que el error típico es de **≈8.7** ventas por día; la validación temporal dice **≈21.2**: ¡más del doble!

¿Por qué? Con los días barajados, el modelo vio el día 300 y el día 302 al entrenar, así que "sabe" en qué nivel estaba la tendencia el día 301: solo **interpola**. En la validación temporal tiene que adivinar la tendencia de días que todavía no han ocurrido, como pasará en la vida real.

El número honesto es el de **TimeSeriesSplit**. Si hubieras reportado 8.7, el modelo te habría decepcionado en producción. (Fíjate también en que el error temporal es mayor en las primeras rondas, cuando el modelo entrenó con pocos datos.)

> 📝 `scoring="neg_mean_absolute_error"` devuelve el MAE **en negativo** porque scikit-learn siempre trata "más alto = mejor". Por eso le cambiamos el signo.
</details>

### ✏️ Ejercicio 9 — ¿Y si quito la columna `dia`?
Alguien sugiere: *"El problema es la columna `dia`; si la quito, el KFold barajado ya no puede hacer trampa."* Cambia la lista de columnas en la celda de abajo y compruébalo. ¿Se acercan los dos resultados? ¿Cuál sigue siendo el honesto?

✍️ *Mi predicción:*

In [ ]:
COLUMNAS = ["dia_semana", "temperatura"]     # ✏️ prueba también ["dia", "dia_semana", "temperatura"]

for nombre, cv in [("KFold barajado ", KFold(5, shuffle=True, random_state=0)),
                   ("TimeSeriesSplit", TimeSeriesSplit(5))]:
    mae = -cross_val_score(bosque, tienda[COLUMNAS], y_t, cv=cv, scoring="neg_mean_absolute_error")
    print(f"{nombre} → MAE promedio: {mae.mean():.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Sin `dia`: KFold barajado **≈32.8**, TimeSeriesSplit **≈38.3**. Ahora sí se acercan: sin el día, el modelo barajado ya no puede "consultar" a los días vecinos para saber el nivel de la tendencia.

Pero fíjate en dos cosas:
- **Los dos errores empeoraron.** El TimeSeriesSplit pasó de ≈21 a ≈38: la columna `dia` sí ayudaba **honestamente** (le dice al modelo en qué etapa reciente está la tendencia). Quitar columnas útiles para que una validación mala parezca buena es tirar información.
- El barajado **sigue siendo algo optimista** (≈33 vs. ≈38), porque otras columnas, como la temperatura, también "delatan" la época del año.

La solución no es quitar columnas: es **validar como se usará el modelo** (con `TimeSeriesSplit`) y conservar las columnas que de verdad ayudan.
</details>

---
## 6. 📖 Validación por grupos: la misma persona no puede estar en los dos lados

Otro tipo de fuga: un hospital tiene **8 visitas** de cada uno de 60 pacientes y quiere predecir un diagnóstico. Si barajas **visitas**, las visitas de la paciente María caen tanto en entrenamiento como en evaluación. El modelo puede aprender a **reconocer a María** (su edad, su peso, sus valores típicos) en lugar de aprender la enfermedad. En producción llegarán **pacientes nuevos**, y ese truco no servirá.

`GroupKFold` mantiene **todas las filas de un mismo grupo** (paciente, cliente, máquina, tienda…) en el mismo pliegue.

### 🔮 Ejercicio 10
En estos datos, el diagnóstico de cada paciente es **pura casualidad**: no depende de sus mediciones (lo hicimos a propósito). Un modelo honesto debería acertar ≈50 %, como lanzar una moneda. ¿Qué exactitud crees que reportará `KFold` barajando visitas? ¿Y `GroupKFold`?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(1)
n_pacientes, visitas = 60, 8
paciente = np.repeat(np.arange(n_pacientes), visitas)            # 0,0,...,0,1,1,...
rasgos = rng.normal(0, 1, (n_pacientes, 5))                       # cada paciente tiene sus valores típicos
X_p = rasgos[paciente] + rng.normal(0, 0.3, (n_pacientes * visitas, 5))
y_p = rng.integers(0, 2, n_pacientes)[paciente]                   # ¡diagnóstico al azar por paciente!

from sklearn.ensemble import RandomForestClassifier
bosque_c = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
print("KFold barajado:", cross_val_score(bosque_c, X_p, y_p, cv=KFold(5, shuffle=True, random_state=0)).mean().round(3))
print("GroupKFold    :", cross_val_score(bosque_c, X_p, y_p, cv=GroupKFold(5), groups=paciente).mean().round(3))

<details><summary><b>👀 Ver respuesta</b></summary>

- **KFold barajado: ≈83 %.** Parece un modelo útil… pero el diagnóstico es aleatorio. El modelo solo reconoce a cada paciente y "recuerda" su diagnóstico de otras visitas.
- **GroupKFold: ≈44 %.** Lo honesto: con pacientes nuevos, el modelo no sabe nada. (Que salga por debajo de 50 % es azar: con solo 60 pacientes, cualquier cosa entre ~40 % y ~60 % es "lanzar una moneda".)

La pregunta clave antes de validar: **¿con qué se usará el modelo en producción?** Si se usará con pacientes (clientes, tiendas, máquinas) **nuevos**, valida por grupos. Si hay grupos **y** clases desbalanceadas, existe `StratifiedGroupKFold`.
</details>

---
## 7. 📖 El flujo completo: la prueba se usa una sola vez

La validación cruzada sirve para **decidir** (qué modelo, qué hiperparámetros). El conjunto de **prueba** sirve para **confirmar**, una sola vez, al final. Si eliges mirando la prueba, la conviertes en otra validación y tu número final deja de ser honesto (Lección 2.5).

```
Todos los datos
 ├── 80 % entrenamiento ──► validación cruzada para comparar y elegir ──► reentrenar el elegido con todo el 80 %
 └── 20 % prueba (bajo llave 🔒) ───────────────────────────────────────► evaluar UNA vez al final
```

▶️ Veámoslo con el diagnóstico de cáncer: elegimos `max_depth` del árbol **solo con validación cruzada**.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

for d in [1, 2, 3, 4, 5, 8]:      # ✏️
    notas_d = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X_tr, y_tr, cv=cv)
    print(f"max_depth = {d}  →  {notas_d.mean():.3f} ± {notas_d.std():.3f}")

### ✋ Ejercicio 11 — Elige y confirma
1. ¿Qué `max_depth` elegirías? Si varios están casi empatados (diferencias menores que la desviación), ¿con cuál te quedas?
2. Pon tu elección en `MEJOR` y ejecuta la celda: es el **examen final**. ¿Se parece al resultado de la validación cruzada?

✍️ *Mis respuestas:*

In [ ]:
MEJOR = 3    # ✏️ tu elección

final = DecisionTreeClassifier(max_depth=MEJOR, random_state=0).fit(X_tr, y_tr)   # reentrena con TODO el entrenamiento
print(f"Exactitud en prueba (una sola vez): {final.score(X_te, y_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. La profundidad 8 tiene el promedio más alto (0.943), pero **todas** las de 2 a 8 están entre 0.927 y 0.943, con desviaciones de ±0.02–0.04: la diferencia es menor que la variación, así que es un **empate**. Ante un empate, el **más simple**: `max_depth` = 2 o 3 (3 tiene el mejor promedio de los simples, 0.930). La de 1 sí queda claramente por debajo (0.903).
2. Con `MEJOR = 3`, la prueba da **0.921**, muy parecido al 0.930 ± 0.036 de la validación cruzada. Así debe ser: si la prueba hubiera salido muy distinta, sospecharías de una fuga o de una división con mala suerte.

⚠️ Ya **no** vale volver atrás, probar `MEJOR = 8` en la prueba y quedarte con el que salga mejor: eso es elegir mirando la prueba.
</details>

> 🔭 **Adelanto del Módulo 7:** este bucle "probar valores con validación cruzada y quedarme con el mejor" es tan común que scikit-learn lo hace en una línea con `GridSearchCV`:
> ```python
> busqueda = GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_depth": [1, 2, 3, 4, 5, 8]}, cv=cv)
> busqueda.fit(X_tr, y_tr)
> busqueda.best_params_, busqueda.best_score_
> ```
> Lo verás a fondo en la Lección 7.3.

---
## 8. 📖 Errores comunes y señales de alerta

| 🚨 Error | Qué pasa | Cómo evitarlo |
|---|---|---|
| Evaluar con una sola división y "creerle" a un número | Lotería: ±5 puntos por pura suerte | Validación cruzada, reportando promedio ± desviación |
| Preprocesar (escalar, seleccionar variables, imputar…) **antes** de dividir | Fuga: el preprocesamiento "vio" los datos de evaluación | Meter el preprocesamiento en un **pipeline** (`make_pipeline`) y validar el pipeline completo |
| Barajar datos con fecha | El modelo ve el futuro: resultado optimista | `TimeSeriesSplit` (y nunca `shuffle=True`) |
| Filas de la misma persona/cliente en entrenamiento y evaluación | El modelo reconoce individuos | `GroupKFold` / `StratifiedGroupKFold` |
| Clases raras con `KFold` al azar | Pliegues con pocos o ningún positivo | `StratifiedKFold` |
| Elegir hiperparámetros mirando la prueba | La prueba deja de ser honesta | Elegir con validación cruzada; la prueba, una vez al final |
| Comparar modelos cuya diferencia es menor que la desviación | Declarar un ganador que no lo es | Mirar la desviación; ante empate, el más simple |

El segundo error es el más traicionero. Veámoslo.

### 🔮 Ejercicio 12 — La fuga invisible
Generamos **ruido puro**: 60 pacientes, 5,000 "mediciones" aleatorias y un diagnóstico al azar. Ningún modelo debería superar ≈50 %. Un analista **primero** elige las 20 mediciones más relacionadas con el diagnóstico (con todos los datos) y **después** hace validación cruzada. Otro mete la selección **dentro de un pipeline**, para que se repita en cada ronda solo con los datos de entrenamiento.

¿Qué exactitud reportará cada uno?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(0)
X_ruido = rng.normal(size=(60, 5000))
y_ruido = rng.integers(0, 2, 60)

# ❌ Analista 1: selecciona con TODOS los datos y luego valida
X_elegidas = SelectKBest(k=20).fit_transform(X_ruido, y_ruido)
print("❌ Seleccionar antes de validar:", cross_val_score(LogisticRegression(), X_elegidas, y_ruido, cv=5).mean().round(3))

# ✅ Analista 2: la selección va DENTRO del pipeline
pipeline = make_pipeline(SelectKBest(k=20), LogisticRegression())
print("✅ Selección dentro del pipeline:", cross_val_score(pipeline, X_ruido, y_ruido, cv=5).mean().round(3))

<details><summary><b>👀 Ver respuesta</b></summary>

- ❌ **≈98 %** con datos que son **puro ruido**. Al elegir las 20 columnas mirando **todas** las filas, el analista eligió justo las que, por casualidad, coinciden con el diagnóstico… incluidas las filas que luego usó para evaluar. La validación cruzada no puede salvarte de una fuga que ocurrió **antes**.
- ✅ **≈58 %**: azar, como debe ser (con solo 60 pacientes, 58 % no se distingue de 50 %).

Regla: **todo lo que "aprende" de los datos** (escalar, imputar, seleccionar, codificar) va **dentro del pipeline**, para que se rehaga en cada ronda solo con el entrenamiento.
</details>

### 🎛️ Perillas de la validación cruzada
| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `n_splits` (k) | Número de pliegues/rondas | Cada modelo entrena con más datos; más lento; notas por pliegue más variables | Más rápido; estimación algo pesimista | 5 (o 10 con pocos datos) |
| `shuffle` | Barajar antes de repartir | — | — | `True` para datos sin orden; **nunca** con series de tiempo |
| `random_state` | Semilla del barajado | — | — | Un número fijo, para que el resultado sea reproducible |
| `gap` (`TimeSeriesSplit`) | Días que se dejan sin usar entre entrenamiento y evaluación | Simula mejor un retraso real (p. ej. los datos tardan una semana en llegar) | Más datos para entrenar | 0, o el retraso real de tu sistema |
| `max_train_size` (`TimeSeriesSplit`) | Tamaño máximo de la ventana de entrenamiento | Usa más historia | Solo historia reciente (útil si el mundo cambia rápido) | Sin límite |
| `groups` | Qué filas deben ir juntas | — | — | El identificador de persona/cliente/máquina, si se repiten |

---
## 9. 🧠 Resumen en 5 puntos
1. Una sola división entrenamiento/prueba es una **lotería**: el mismo modelo puede variar 10 puntos según qué filas caen en cada lado.
2. La **validación cruzada k-fold** rota el papel de evaluación entre k pliegues y reporta **promedio ± desviación**. k = 5 o 10 es el estándar.
3. Elige la variante según **cómo se usará el modelo**: **estratificada** con clases raras, **temporal** con datos fechados (entrenar con el pasado, evaluar con el futuro), **por grupos** si se repiten personas o clientes.
4. La validación cruzada sirve para **decidir**; el conjunto de **prueba**, para **confirmar una sola vez** al final.
5. Todo preprocesamiento que aprende de los datos va **dentro de un pipeline**; si no, la validación cruzada mide una fuga, no tu modelo.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈35 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** Con 2,000 filas y validación cruzada de 10 pliegues, ¿con cuántas filas se entrena cada modelo?
- a) 200 · b) 1,000 · c) 1,800 · d) 2,000

**2.** Quieres predecir la demanda de energía de **mañana** con datos de los últimos 3 años. ¿Qué validación usas?
- a) `KFold(shuffle=True)` · b) `StratifiedKFold` · c) `TimeSeriesSplit` · d) `GroupKFold`

**3.** Tu modelo A saca 0.861 ± 0.020 y el B 0.866 ± 0.022 en validación cruzada. B es mucho más complejo. ¿Qué haces?
- a) Elegir B: tiene el promedio más alto · b) Elegir A: la diferencia es mucho menor que la variación y A es más simple · c) Evaluar los dos en la prueba y quedarte con el mejor · d) Subir k hasta que B gane claramente

**4.** Un dataset de radiografías tiene **varias imágenes por paciente**. Para estimar cómo funcionará con pacientes nuevos usas…
- a) `KFold` · b) `GroupKFold` con el ID del paciente · c) `TimeSeriesSplit` · d) Una sola división al azar

**5.** ¿Cuál de estas acciones **no** provoca fuga de datos?
- a) Escalar todo el dataset y luego hacer validación cruzada · b) Seleccionar variables con todo el dataset y luego validar · c) Meter el escalado dentro de un pipeline y validar el pipeline · d) Elegir el hiperparámetro que mejor sale en la prueba

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Cada ronda deja fuera 1/10 (200 filas) y entrena con 1,800.
2. **c)** Datos fechados y se predice el futuro: hay que validar respetando el tiempo.
3. **b)** 0.005 de diferencia frente a ±0.02 de variación: es un empate. Ante un empate, el modelo más simple. La c) usa la prueba para decidir (trampa) y la d) es buscar el resultado que quieres.
4. **b)** Si no, las imágenes de un mismo paciente caen en los dos lados y el modelo reconoce pacientes en lugar de enfermedades.
5. **c)** Es justo la forma correcta: el escalado se aprende en cada ronda solo con el entrenamiento.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Un equipo predice la **rotación de empleados** de una empresa con datos de 5 años: cada fila es un empleado-año (el mismo empleado aparece hasta 5 veces). Quieren saber qué tan bien funcionará el modelo **el próximo año**. ¿Qué problemas tendría un `KFold` barajado? ¿Cómo diseñarías la validación?

**B. Caso de análisis 💼** Una startup te presenta su modelo: *"98.7 % de exactitud en validación cruzada"*. ¿Qué **cuatro preguntas** les harías antes de creerles? (Piensa en todo el módulo: métricas, desbalance, fugas, tipo de validación.)

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Dos fugas a la vez: (1) **temporal**: entrenaría con el año 5 y evaluaría con el año 3, viendo el futuro (por ejemplo, una crisis que hizo que muchos se fueran); (2) **por grupos**: el mismo empleado estaría en los dos lados. Diseño razonable: validación **temporal por años**: entrenar con los años 1–2 y evaluar en el 3; luego 1–3 → 4; luego 1–4 → 5. Así se imita exactamente "entrenar con lo que sé, predecir el año que viene". Como los años de evaluación son posteriores, la fuga por grupos se reduce mucho, aunque un mismo empleado siga apareciendo en entrenamiento y evaluación: es aceptable, porque en producción el modelo **sí** habrá visto a los empleados actuales en años anteriores.

**B.** Por ejemplo:
1. ¿Qué tan desbalanceadas están las clases? Si el 98 % es de una clase, decir siempre esa clase da 98 % (Lecciones 4.2 y 4.7). ¿Cuánto da una línea base?
2. ¿Qué *recall* y precisión tiene en la clase que importa? (Lecciones 4.2–4.3.)
3. ¿Cómo dividieron? ¿Hay fechas (¿validación temporal?) o filas repetidas de la misma persona (¿por grupos?)?
4. ¿El preprocesamiento y la selección de variables estaban **dentro** del pipeline? ¿Alguna variable se conoce solo **después** del resultado que se predice?
5. (Extra) ¿Cuál es la desviación entre pliegues? ¿Cuánto da en un conjunto de prueba que nunca tocaron?
</details>

### 10.3 🧩 Práctica integradora: "¿Completará su rutina?" 🏃

Una app de ejercicio quiere predecir si un usuario **completará** la rutina que acaba de empezar, para mandarle un mensaje de ánimo a quien probablemente la abandonará. Tiene 1,800 sesiones de 150 usuarios. Como solo ~23 % de las sesiones se completan, usaremos el **AUC** (Lección 4.3) como métrica.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos y aparta la prueba**

In [ ]:
rng = np.random.default_rng(5)
n_usuarios, sesiones = 150, 12
u = np.repeat(np.arange(n_usuarios), sesiones)
edad = rng.integers(18, 65, n_usuarios)
peso = rng.normal(75, 12, n_usuarios).round(1)
constancia = rng.normal(0, 1.5, n_usuarios)                 # rasgo de cada persona que la app NO mide
duracion = rng.choice([20, 30, 45, 60, 90], n_usuarios * sesiones)
hora = rng.integers(6, 23, n_usuarios * sesiones)
dias_sin = rng.integers(0, 8, n_usuarios * sesiones)
z = -1.5 + constancia[u] - 0.025 * (duracion - 45) - 0.3 * (dias_sin - 3) + 0.02 * (edad[u] - 40)
app = pd.DataFrame({"usuario": u + 1000, "edad": edad[u], "peso_kg": peso[u], "duracion_min": duracion,
                    "hora": hora, "dias_sin_entrenar": dias_sin,
                    "completo": (rng.random(len(u)) < 1 / (1 + np.exp(-z))).astype(int)})

X_app = app.drop(columns=["usuario", "completo"]); y_app = app["completo"]; grupos = app["usuario"]
# 🔒 La prueba: 25 % de los USUARIOS (no de las sesiones), guardada bajo llave hasta el paso 5
entrena, prueba = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(X_app, y_app, grupos))
X_tr_a, X_te_a = X_app.iloc[entrena], X_app.iloc[prueba]
y_tr_a, y_te_a = y_app.iloc[entrena], y_app.iloc[prueba]
g_tr = grupos.iloc[entrena]

print(f"Sesiones: {len(app)} · usuarios: {app.usuario.nunique()} · completadas: {y_app.mean():.0%}")
print(f"Entrenamiento: {len(X_tr_a)} sesiones de {g_tr.nunique()} usuarios · Prueba: {len(X_te_a)} sesiones")
app.head()

✋ Responde: ¿por qué apartamos la prueba por **usuarios** y no por sesiones? ¿Qué tipo de validación cruzada exigen estos datos?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque la app usará el modelo con **usuarios nuevos** que se acaban de registrar. Cada usuario tiene 12 sesiones con la **misma** edad y el **mismo** peso: si sus sesiones caen en los dos lados, el modelo puede reconocerlo y "recordar" si suele terminar sus rutinas. Por eso la validación debe ser **por grupos** y, como la clase positiva es minoritaria (23 %), conviene que sea también **estratificada**: `StratifiedGroupKFold`.
</details>

**Paso 2 🔮 — Validación ingenua vs. por grupos.** Evaluamos un árbol con `max_depth=8` de las dos formas. ¿Cuál dará un AUC más alto? ¿Por cuánto?

✍️ *Mi apuesta:*

In [ ]:
cv_ingenua = StratifiedKFold(5, shuffle=True, random_state=0)
cv_grupos = StratifiedGroupKFold(5, shuffle=True, random_state=0)
arbol8 = DecisionTreeClassifier(max_depth=8, random_state=0)

auc_ing = cross_val_score(arbol8, X_tr_a, y_tr_a, cv=cv_ingenua, scoring="roc_auc").mean()
auc_gru = cross_val_score(arbol8, X_tr_a, y_tr_a, cv=cv_grupos, groups=g_tr, scoring="roc_auc").mean()
print(f"AUC con StratifiedKFold (sesiones barajadas): {auc_ing:.3f}")
print(f"AUC con StratifiedGroupKFold (por usuario)  : {auc_gru:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Barajando sesiones: **0.686**.
- Por usuario: **0.592**.

Casi **0.1 de AUC** de diferencia: la validación ingenua deja que el árbol reconozca a cada usuario por su edad y su peso (que se repiten en sus 12 sesiones) y "recuerde" su constancia, el rasgo que la app no mide. Con usuarios nuevos, ese truco no existe.
</details>

**Paso 3 ✏️ — Elige `max_depth` de las dos formas.** Ejecuta la tabla. ¿Qué profundidad elegiría cada validación? ¿Coinciden?

In [ ]:
filas = []
for d in [1, 2, 3, 4, 5, 6, 8, 10, None]:     # ✏️
    arbol = DecisionTreeClassifier(max_depth=d, random_state=0)
    filas.append({"max_depth": str(d),
                  "AUC ingenua": cross_val_score(arbol, X_tr_a, y_tr_a, cv=cv_ingenua, scoring="roc_auc").mean(),
                  "AUC por grupos": cross_val_score(arbol, X_tr_a, y_tr_a, cv=cv_grupos, groups=g_tr, scoring="roc_auc").mean()})
pd.DataFrame(filas).set_index("max_depth").round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

- La validación **ingenua** elegiría `max_depth` = **6 u 8** (0.686).
- La validación **por grupos** elegiría **5** (0.648).

No coinciden, y fíjate en **la brecha** entre columnas: con `max_depth=2` es pequeña (0.613 vs. 0.592) y con `max_depth=8` es enorme (0.686 vs. 0.592). Cuanto más profundo el árbol, más capaz es de memorizar usuarios, y la validación ingenua **premia** esa memorización. Una mala validación no solo da un número inflado: te hace **elegir mal** el modelo.
</details>

**Paso 4 ✋ — Tu decisión.** Escribe en una frase qué `max_depth` eliges y **con qué validación** lo justificas.

✍️ *Mi decisión:*

<details><summary><b>👀 Ver respuesta</b></summary>

*"Elijo `max_depth=5` porque es el que mejor sale en validación cruzada **por usuarios** (StratifiedGroupKFold), que imita lo que pasará en producción: predecir para personas que el modelo no ha visto."* (Si eliges 4, con 0.626, también es defendible: es más simple y la diferencia es pequeña.)
</details>

**Paso 5 ▶️ — Examen final, una sola vez.** Ahora sí abrimos la prueba (usuarios que el modelo **nunca** vio). Compara el resultado con lo que prometía cada validación.

In [ ]:
ELEGIDO = 5     # ✏️ tu max_depth

final_app = DecisionTreeClassifier(max_depth=ELEGIDO, random_state=0).fit(X_tr_a, y_tr_a)
from sklearn.metrics import roc_auc_score
print(f"AUC en prueba (usuarios nuevos): {roc_auc_score(y_te_a, final_app.predict_proba(X_te_a)[:, 1]):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

La prueba da **0.584**. Compáralo con lo que prometía cada validación para `max_depth=5`:
- la **ingenua**: 0.661 (y su favorito, `max_depth=8`, prometía 0.686);
- la **por grupos**: 0.648.

Las dos se quedaron cortas, pero la ingenua mucho más. La prueba tiene solo 38 usuarios, así que también "tiembla" unos cuantos puntos (Sección 2): por eso **no** se usa para volver a elegir. Lo que sí te dice es la realidad: el modelo es **débil** (AUC ≈0.6, poco mejor que el azar de 0.5), porque lo más importante, la constancia de cada persona, la app no lo mide. Esa conclusión honesta vale más que un 0.686 inflado.
</details>

**Paso 6 ✋ — Explícaselo a la gerente de producto.** Ella vio en una presentación anterior que "el modelo tiene un AUC de casi 0.69". Explícale en 3 frases, sin palabras técnicas, por qué tu número es más bajo y por qué es **el que debe creer**.

✍️ *Mi explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"El número anterior se midió con personas que el modelo ya conocía: había visto otras sesiones suyas y en parte las reconocía. Nuestro número se midió como se usará la app de verdad, con usuarios que el modelo nunca había visto, y por eso es más bajo. Es menos espectacular, pero es el que se va a cumplir cuando lancemos los mensajes."* Y un extra útil: *"Para mejorar, la app necesita medir algo que hoy no mide, como qué tan constante ha sido la persona en sus primeras semanas."*
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, guía de usuario, *Cross-validation: evaluating estimator performance* (incluye un diagrama de cada estrategia: `KFold`, `StratifiedKFold`, `GroupKFold`, `TimeSeriesSplit`…).
- 📘 **Documentación:** scikit-learn, *Visualizing cross-validation behavior in scikit-learn* (galería de ejemplos con gráficas de los pliegues).
- 📺 **Video:** StatQuest, "Machine Learning Fundamentals: Cross Validation" (inglés, con subtítulos).
- 📘 **Libro:** *Forecasting: Principles and Practice* (Hyndman y Athanasopoulos), sección *Time series cross-validation*. Gratis en línea.

---
⬅️ **Anterior:** [4.5 El compromiso sesgo-varianza: la analogía del tirador al blanco](4.5-sesgo-varianza.ipynb) · ➡️ **Siguiente:** [4.7 Líneas base: ¿le gano a "adivinar el promedio"?](4.7-lineas-base.ipynb)